<a href="https://colab.research.google.com/github/mejia080902-bit/simulacion_II/blob/main/simulacion_II_muestreo_importancia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Simulación II - Comparación de eficiencia:
Monte Carlo Crudo vs Muestreo por Importancia
Integral: I = ∫_0^1 cos(pi*x/2) dx = 2/pi
"""

import numpy as np
import time

# -------------------------
# Función de interés y valor verdadero
# -------------------------
def g(x):
    # g(x) = cos(pi*x/2)
    return np.cos(np.pi * x / 2.0)

I_true = 2.0 / np.pi  # valor analítico para referencia

# -------------------------
# Método Monte Carlo Crudo (Uniforme[0,1])
# -------------------------
def crudo(N, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    x = rng.uniform(0.0, 1.0, size=N)
    gx = g(x)
    mean = gx.mean()
    var = gx.var(ddof=1)  # varianza por-muestra (no dividida entre N)
    std = var**0.5
    return mean, var, std, gx

# -------------------------
# Muestreo por Importancia con Beta(1, b)
# p_b(x) = b * (1-x)^(b-1), x in [0,1]
# -------------------------
def importancia_beta_1b(N, b=3.0, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    x = rng.beta(1.0, b, size=N)
    p = b * (1.0 - x)**(b - 1.0)
    w = g(x) / p              # pesos por-muestra
    est = w.mean()            # estimador IS
    var_w = w.var(ddof=1)     # varianza por-muestra de w
    std_w = var_w**0.5
    return est, var_w, std_w, {"b": b, "samples": x}

# -------------------------
# Pilot para elegir b (minimiza varianza por-muestra de w)
# -------------------------
def elegir_b_pilot(rng, b_grid=(1.5, 2, 3, 4, 5, 6), N_pilot=2000):
    mejores = []
    for b in b_grid:
        _, var_w, _, _ = importancia_beta_1b(N_pilot, b=b, rng=rng)
        mejores.append((var_w, b))
    mejores.sort()
    return mejores[0][1]  # b con menor var_w

# -------------------------
# Ejecución y tiempos
# -------------------------
if __name__ == "__main__":
    N = 10_000
    rng = np.random.default_rng(12345)

    # Crudo
    t0 = time.time()
    mean_crudo, var_crudo, std_crudo, gx = crudo(N, rng)
    t_crudo = time.time() - t0

    # Elegir b por pilot
    b_opt = elegir_b_pilot(rng, b_grid=(1.5, 2, 3, 4, 5, 6), N_pilot=3000)

    # Importancia (con b óptimo del pilot)
    t0 = time.time()
    est_is, var_is, std_is, info_is = importancia_beta_1b(N, b=b_opt, rng=rng)
    t_is = time.time() - t0

    # -------------------------
    # Resultados
    # -------------------------
    print("Valor verdadero I = 2/pi = {:.8f}\n".format(I_true))

    print("---- Método Crudo (Unif[0,1]) ----")
    print(f"Estimador = {mean_crudo:.8f}, Var = {var_crudo:.6e}, Std = {std_crudo:.6e}, Tiempo = {t_crudo:.5f} s")

    print("\n---- Muestreo por Importancia (Beta(1,b)) ----")
    print(f"b (óptimo pilot) = {info_is['b']}")
    print(f"Estimador = {est_is:.8f}, Var(w) = {var_is:.6e}, Std(w) = {std_is:.6e}, Tiempo = {t_is:.5f} s")

    # -------------------------
    # Comparaciones
    #   - Reducción de varianza (por muestra)
    #   - Eficiencia relativa ajustada por tiempo: ε = (t * Var)IS / (t * Var)Crudo
    # -------------------------
    reduccion = 100.0 * (1.0 - (var_is / var_crudo))
    eps_is = (t_is * var_is) / (t_crudo * var_crudo)

    print("\n---- Comparación ----")
    print(f"Reducción de varianza (IS vs Crudo): {reduccion:.2f}%")
    print(f"Eficiencia relativa ε(IS vs Crudo)  : {eps_is:.6f}  (ε < 1 ⇒ IS más eficiente)")

    # (Opcional) errores estándar de la MEDIA (dividiendo por N)
    se_crudo = (var_crudo / N) ** 0.5
    se_is    = (var_is    / N) ** 0.5
    print(f"\nError estándar de la media (Crudo): {se_crudo:.6e}")
    print(f"Error estándar de la media (IS)   : {se_is:.6e}")


Valor verdadero I = 2/pi = 0.63661977

---- Método Crudo (Unif[0,1]) ----
Estimador = 0.64204022, Var = 9.313492e-02, Std = 3.051801e-01, Tiempo = 0.00069 s

---- Muestreo por Importancia (Beta(1,b)) ----
b (óptimo pilot) = 2
Estimador = 0.63599183, Var(w) = 6.778710e-03, Std(w) = 8.233292e-02, Tiempo = 0.00092 s

---- Comparación ----
Reducción de varianza (IS vs Crudo): 92.72%
Eficiencia relativa ε(IS vs Crudo)  : 0.097045  (ε < 1 ⇒ IS más eficiente)

Error estándar de la media (Crudo): 3.051801e-03
Error estándar de la media (IS)   : 8.233292e-04
